In [1]:
import warp as wp
from icecream import ic


In [ ]:
a = wp.array([wp.vec3(1.0, 0.0, 3.0)],dtype=wp.vec3, requires_grad=True)
b = wp.array([wp.vec3(1.0, 2.0, 0.0)],dtype=wp.vec3, requires_grad=True)
c = wp.zeros(1024, dtype=wp.vec3, device="cuda", requires_grad=True)
result = wp.zeros(1, dtype=float, device="cuda", requires_grad=True)
print(a)


Warp 1.6.2 initialized:
   CUDA Toolkit 12.8, Driver 12.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (24 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/grablab/.cache/warp/1.6.2
[[1. 0. 3.]]


In [28]:
tu = wp.vec3(1.0, 0.0, 3.0)
ti = wp.vec3(1.0, 2.0, 0.0)

result = tu - ti
print(result)
loss = wp.length(result)
print(loss)

[0.0, -2.0, 3.0]
3.605551242828369


In [3]:
@wp.kernel
def compute1(
    a: wp.array(dtype=wp.vec3),
    b: wp.array(dtype=wp.vec3),
    c: wp.array(dtype=wp.vec3),
    result: wp.array(dtype=float),
):
    tid = wp.tid()
    result[tid] = wp.dot(a[tid], b[tid])

In [4]:
tape = wp.Tape()

# forward pass
with tape:
    wp.launch(kernel=compute1,
              dim = 1,
              inputs=(a, b,c,result),
              device="cuda"
              )

# reverse pass
tape.backward(result)
print(a.grad)

Module __main__ cb0b494 load on device 'cuda:0' took 0.28 ms  (cached)
[[1. 2. 0.]]


In [17]:
import torch

In [18]:
x = torch.tensor([1.,2.,3.],requires_grad=True)
y = torch.tensor([0.,2.,0.],requires_grad=True)

In [19]:
dot = x + y
ic(dot)

dot.backward
ic(y.grad)

ic| dot: tensor([1., 4., 3.], grad_fn=<AddBackward0>)
ic| y.grad: None


In [23]:
import torch

x = torch.tensor([1.,2.,3.],requires_grad=True)
y = torch.tensor([0.,2.,0.],requires_grad=True)
dot = x + y
ic(dot)
ic(torch.ones_like(dot))
dot.backward(torch.ones_like(dot))
ic(y.grad)

ic| dot: tensor([1., 4., 3.], grad_fn=<AddBackward0>)
ic| torch.ones_like(dot): tensor([1., 1., 1.])
ic| y.grad: tensor([1., 1., 1.])


tensor([1., 1., 1.])